# LeetCode 894 — All Possible Full Binary Trees bằng Python

Link đề: https://leetcode.com/problems/all-possible-full-binary-trees/ · Độ khó: **Medium** · Chủ đề: Tree (cây), Dynamic Programming (quy hoạch động), Backtracking (quay lui), Memoization (ghi nhớ), Binary Tree (cây nhị phân)

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def allPossibleFBT(self, n: int) -> List[Optional[TreeNode]]:
        ...
```

Ghi chú: một **cây nhị phân đầy đủ** (full binary tree) là cây mà mỗi nút có đúng `0` hoặc `2` con. Bài yêu cầu **liệt kê mọi cây** có `n` nút (giá trị mỗi nút là `0`), theo **thứ tự bất kỳ**. Vì vậy khi kiểm thử ta **tự dựng `TreeNode`**, **tuần tự hóa** mỗi cây thành dạng so sánh được rồi so sánh **tập** các cây sau khi `sorted`. Một nhận xét nhanh: số nút của cây đầy đủ luôn **lẻ** (`1 + 2k`), nên `n` chẵn cho danh sách rỗng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: liệt kê **tất cả** cây nhị phân đầy đủ có đúng `n` nút (mỗi nút giá trị `0`), mỗi phần tử trả về là gốc của một cây.
2. Nắm các công cụ Python cần thiết: lớp `TreeNode`, đệ quy trả về **danh sách cây**, `itertools.product` hoặc hai vòng `for` lồng nhau để ghép mọi cặp (cây trái, cây phải), `functools.cache` để ghi nhớ, và tuần tự hóa cây để so sánh.
3. Hiểu nguyên lý chia để trị: cây `n` nút gồm **một gốc + cây con trái `i` nút + cây con phải `n - 1 - i` nút**, trong đó `i` và `n - 1 - i` đều **lẻ** (vì cả hai cây con đều đầy đủ); mỗi cặp (cây trái, cây phải) cho một cây mới.
4. Đi qua 4 cách: liệt kê mọi hình dạng cây rồi lọc (brute), đệ quy không nhớ, đệ quy có ghi nhớ (chia sẻ cây con, cách nộp chính) và quy hoạch động từ dưới lên theo số nút; đối chiếu bằng oracle **mở rộng từng lá** và công thức số **Catalan**.
5. Tránh các lỗi kinh điển: cho phép cây con rỗng (tạo nút một con), sai cận vòng lặp, sửa nút trong cây khi các cây **chia sẻ** cây con, và so sánh cây bằng `==` (không tuần tự hóa).

## 2. Đề bài & ví dụ

Cho số nguyên `n`. Trả về **danh sách mọi cây nhị phân đầy đủ** có `n` nút. Mỗi nút của mỗi cây có `Node.val == 0`.

Mỗi phần tử của kết quả là **nút gốc** của một cây. Có thể trả về theo **thứ tự bất kỳ**.

Một **cây nhị phân đầy đủ** là cây nhị phân mà mỗi nút có đúng `0` hoặc `2` con.

**Ví dụ 1**

- Input: `n = 7`
- Output: `[[0,0,0,null,null,0,0,null,null,0,0],[0,0,0,null,null,0,0,0,0],[0,0,0,0,0,0,0],[0,0,0,0,0,null,null,null,null,0,0],[0,0,0,0,0,null,null,0,0]]` (5 cây)

**Ví dụ 2**

- Input: `n = 3`
- Output: `[[0,0,0]]`

**Ràng buộc:**

- `1 <= n <= 20`

In [1]:
from typing import List, Optional
from functools import cache
from itertools import product
from math import comb
import random, time, sys

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(vals: List[Optional[int]]) -> Optional[TreeNode]:
    """Dựng cây từ danh sách theo mức kiểu LeetCode (None là nút trống)."""
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

def tree_to_list(root: Optional[TreeNode]) -> List[Optional[int]]:
    """Xuất cây theo mức kiểu LeetCode (bỏ None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

def serialize(root: Optional[TreeNode]) -> str:
    """Tuần tự hóa tiền thứ tự có dấu '#' cho nút trống: duy nhất cho mỗi hình dạng (kèm giá trị)."""
    if root is None:
        return "#"
    return f"{root.val}({serialize(root.left)},{serialize(root.right)})"

def to_shape(root: Optional[TreeNode]):
    """Hình dạng thuần túy (bỏ giá trị): () là lá, (L, R) là nút trong."""
    if root is None:
        return None
    return (to_shape(root.left), to_shape(root.right))

def count_nodes(root: Optional[TreeNode]) -> int:
    return 0 if root is None else 1 + count_nodes(root.left) + count_nodes(root.right)

def is_full(root: Optional[TreeNode]) -> bool:
    if root is None:
        return False
    if root.left is None and root.right is None:
        return True
    return root.left is not None and root.right is not None and is_full(root.left) and is_full(root.right)

examples = [
    (7, [[0, 0, 0, None, None, 0, 0, None, None, 0, 0], [0, 0, 0, None, None, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, None, None, None, None, 0, 0], [0, 0, 0, 0, 0, None, None, 0, 0]]),
    (3, [[0, 0, 0]]),
]
for n, expected in examples:
    print(f"n={n}: {len(expected)} cây kỳ vọng; ví dụ cây đầu: {expected[0]}")
    for lst in expected:
        assert is_full(build_tree(lst)) and count_nodes(build_tree(lst)) == n

n=7: 5 cây kỳ vọng; ví dụ cây đầu: [0, 0, 0, None, None, 0, 0, None, None, 0, 0]
n=3: 1 cây kỳ vọng; ví dụ cây đầu: [0, 0, 0]


## 3. Kiến thức nền

### 3.1. Cây nhị phân đầy đủ và số nút lẻ

Mỗi nút có `0` hoặc `2` con. Nếu cây có `k` nút trong (nút có 2 con) thì có `k + 1` lá, tổng số nút là `2k + 1`: **luôn lẻ**. Hệ quả: `n` chẵn thì không có cây nào (trả `[]`).

In [2]:
def tree_with_internal(k: int) -> TreeNode:
    """Một cây đầy đủ có k nút trong: xâu chuỗi trái."""
    root = TreeNode()
    cur = root
    for _ in range(k):
        cur.left, cur.right = TreeNode(), TreeNode()
        cur = cur.left
    return root

for k in range(0, 5):
    t = tree_with_internal(k)
    print(f"k={k} nút trong -> tổng số nút {count_nodes(t)} (= 2k + 1 = {2 * k + 1}), đầy đủ? {is_full(t)}")

k=0 nút trong -> tổng số nút 1 (= 2k + 1 = 1), đầy đủ? True
k=1 nút trong -> tổng số nút 3 (= 2k + 1 = 3), đầy đủ? True
k=2 nút trong -> tổng số nút 5 (= 2k + 1 = 5), đầy đủ? True
k=3 nút trong -> tổng số nút 7 (= 2k + 1 = 7), đầy đủ? True
k=4 nút trong -> tổng số nút 9 (= 2k + 1 = 9), đầy đủ? True


### 3.2. Chia để trị: gốc + trái + phải

Cây đầy đủ `n` nút (với `n > 1`) có gốc, một cây con **trái** đầy đủ `i` nút và một cây con **phải** đầy đủ `n - 1 - i` nút. Vì mỗi cây con là đầy đủ nên cả `i` và `n - 1 - i` đều **lẻ** (và `>= 1`: nếu một bên rỗng thì gốc chỉ có một con, vi phạm). Với `n = 7`: `(i, n-1-i) = (1, 5), (3, 3), (5, 1)`.

In [3]:
n = 7
print([(i, n - 1 - i) for i in range(1, n - 1, 2)])

[(1, 5), (3, 3), (5, 1)]


### 3.3. Tuần tự hóa để so sánh cây

Hai `TreeNode` khác đối tượng không bằng nhau theo `==` ngay cả khi cùng cấu trúc (lớp không định nghĩa `__eq__`). Để so sánh **tập** các cây ta chuyển mỗi cây thành chuỗi `serialize(root)` rồi so sánh tập hoặc danh sách đã sắp xếp.

In [4]:
a, b = TreeNode(), TreeNode()
print("a == b ?", a == b, "| serialize:", serialize(a), serialize(b), serialize(a) == serialize(b))
t = build_tree([0, 0, 0, None, None, 0, 0])
print(serialize(t), "| danh sách theo mức:", tree_to_list(t))

a == b ? False | serialize: 0(#,#) 0(#,#) True
0(0(#,#),0(0(#,#),0(#,#))) | danh sách theo mức: [0, 0, 0, None, None, 0, 0]


### 3.4. Số Catalan

Số cây nhị phân đầy đủ có `n = 2m + 1` nút là số Catalan thứ `m`: `C_m = C(2m, m) / (m + 1)`: `1, 1, 2, 5, 14, 42, 132, 429, 1430, 4862, ...` cho `n = 1, 3, 5, 7, 9, 11, ...`. Số này lớn lên rất nhanh, nên bộ nhớ và thời gian của mọi cách đều tỷ lệ với số cây cần liệt kê.

In [5]:
def catalan(m: int) -> int:
    return comb(2 * m, m) // (m + 1)

print([(2 * m + 1, catalan(m)) for m in range(10)])

[(1, 1), (3, 1), (5, 2), (7, 5), (9, 14), (11, 42), (13, 132), (15, 429), (17, 1430), (19, 4862)]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ `n = 5` (có `C_2 = 2` cây):

- Gốc `0`. Chia `n - 1 = 4` nút còn lại thành (trái, phải) lẻ: `(1, 3)` và `(3, 1)`.
- `(1, 3)`: cây trái là một nút; cây phải có `1` cây đầy đủ 3 nút (gốc + hai lá). Ra cây: gốc, trái là lá, phải là cây 3 nút.
- `(3, 1)`: đối xứng: trái là cây 3 nút, phải là lá.

Vậy `n = 5` có `1 * 1 + 1 * 1 = 2` cây. Với `n = 7`: `(1,5)`: `1 * 2 = 2` cây; `(3,3)`: `1 * 1 = 1`; `(5,1)`: `2 * 1 = 2`; tổng `5` cây. Đúng với ví dụ 1.

## 5. Cách 1 — Brute Force: liệt kê mọi hình dạng cây rồi lọc

**Ý tưởng.** Sinh **mọi cây nhị phân** có `n` nút (không cần đầy đủ): với số nút ở trái `i` từ `0` đến `n - 1`, ghép mọi hình dạng cây trái `i` nút với mọi hình dạng cây phải `n - 1 - i` nút (số Catalan `C_n`: `n = 11` đã có `58786`). Sau đó **lọc** các cây mà mỗi nút có `0` hoặc `2` con. Đây là định nghĩa trực tiếp nên cũng là **oracle** cho `n` nhỏ.

In [6]:
@cache
def all_binary_shapes(n: int) -> tuple:
    """Mọi hình dạng cây nhị phân có n nút (None là rỗng; nút là cặp (trái, phải))."""
    if n == 0:
        return (None,)
    out = []
    for i in range(n):
        for l in all_binary_shapes(i):
            for r in all_binary_shapes(n - 1 - i):
                out.append((l, r))
    return tuple(out)

def shape_is_full(sh) -> bool:
    if sh is None:
        return False
    l, r = sh
    if l is None and r is None:
        return True
    return l is not None and r is not None and shape_is_full(l) and shape_is_full(r)

def full_shapes_brute(n: int) -> set:
    return {sh for sh in all_binary_shapes(n) if shape_is_full(sh)}

def shape_to_tree(sh) -> Optional[TreeNode]:
    if sh is None:
        return None
    return TreeNode(0, shape_to_tree(sh[0]), shape_to_tree(sh[1]))

for n, expected in examples:
    got = full_shapes_brute(n)
    print(f"n={n}: {len(got)} cây hình dạng đầy đủ (kỳ vọng {len(expected)})")
    assert got == {to_shape(build_tree(lst)) for lst in expected}

n=7: 5 cây hình dạng đầy đủ (kỳ vọng 5)
n=3: 1 cây hình dạng đầy đủ (kỳ vọng 1)


**Độ phức tạp và nút thắt.** Số cây nhị phân tổng quát là số Catalan `C_n`, lớn hơn nhiều số cây đầy đủ. Xem số tăng và thời gian:

In [7]:
for n in (3, 5, 7, 9, 11, 13):
    t0 = time.perf_counter()
    total = len(all_binary_shapes(n))
    full = len(full_shapes_brute(n))
    print(f"n={n:2}: {total:7} cây nhị phân tổng quát, chỉ {full:3} cây đầy đủ, {1000 * (time.perf_counter() - t0):8.1f} ms")

n= 3:       5 cây nhị phân tổng quát, chỉ   1 cây đầy đủ,      0.0 ms
n= 5:      42 cây nhị phân tổng quát, chỉ   2 cây đầy đủ,      0.0 ms
n= 7:     429 cây nhị phân tổng quát, chỉ   5 cây đầy đủ,      0.0 ms
n= 9:    4862 cây nhị phân tổng quát, chỉ  14 cây đầy đủ,      1.3 ms
n=11:   58786 cây nhị phân tổng quát, chỉ  42 cây đầy đủ,     17.2 ms


n=13:  742900 cây nhị phân tổng quát, chỉ 132 cây đầy đủ,    235.6 ms


Phần lớn các hình dạng bị loại (chỉ `~C_{n/2}` trong số `C_n`). Nút thắt: ta sinh cây **không đầy đủ** rồi bỏ đi; thay vào đó có thể **chỉ sinh** các cây có cây con trái/phải đầy đủ ngay từ đầu.

## 6. Ý tưởng thuật toán / bước đột phá

**Cây đầy đủ được tạo từ các cây đầy đủ nhỏ hơn.** Nếu `F(n)` là danh sách mọi cây đầy đủ `n` nút, thì với `n > 1`:

```
F(n) = [ Node(0, l, r)  for i in 1, 3, 5, ..., n - 2
                        for l in F(i)  for r in F(n - 1 - i) ]
F(1) = [ Node(0) ]
F(n chẵn) = []
```

Mỗi cây trong `F(n)` tương ứng **đúng một** bộ ba (kích thước cây trái, cây trái, cây phải), nên không trùng và không bỏ sót.

| Cách | Ý tưởng |
|---|---|
| 2 | Đệ quy theo công thức, **dựng cây mới** mỗi lần (không chia sẻ nút) |
| 3 | Thêm **ghi nhớ** `F(k)`: các cây con được **chia sẻ** giữa nhiều cây (nộp) |
| 4 | Quy hoạch động **từ dưới lên**: tính `F(1), F(3), F(5), ...` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đệ quy không ghi nhớ (dựng cây mới mỗi lần)

**Giải thích từng đoạn.**

- `n % 2 == 0`: trả `[]` (không có cây).
- `n == 1`: trả `[TreeNode()]`.
- Với mỗi `i` lẻ trong `1, 3, ..., n - 2`: gọi đệ quy cho `i` và `n - 1 - i`, rồi với mỗi cặp `(l, r)` tạo `TreeNode(0, l, r)`.
- Với mỗi `i`, gọi đệ quy **một lần** cho cây trái (`lefts`) và **một lần** cho cây phải (`rights`), rồi ghép mọi cặp; cây con được dùng chung giữa các cặp trong cùng vòng lặp.
- Không có bộ nhớ giữa các nhánh: cùng một `F(k)` bị **tính lại** ở nhiều nơi trong cây đệ quy (ví dụ `F(1)` và `F(3)` xuất hiện nhiều lần).

In [8]:
def fbt_recursive(n: int) -> List[TreeNode]:
    if n % 2 == 0:
        return []
    if n == 1:
        return [TreeNode()]
    res = []
    for i in range(1, n - 1, 2):
        lefts, rights = fbt_recursive(i), fbt_recursive(n - 1 - i)
        for l in lefts:
            for r in rights:
                res.append(TreeNode(0, l, r))
    return res

for n, expected in examples:
    got = fbt_recursive(n)
    assert sorted(serialize(t) for t in got) == sorted(serialize(build_tree(lst)) for lst in expected)
    print(f"n={n}: {len(got)} cây khớp ví dụ")

n=7: 5 cây khớp ví dụ
n=3: 1 cây khớp ví dụ


**Code nộp LeetCode (Cách 2):**

In [9]:
class Solution:
    def allPossibleFBT(self, n: int) -> List[Optional[TreeNode]]:
        if n % 2 == 0:
            return []
        if n == 1:
            return [TreeNode()]
        res = []
        for i in range(1, n - 1, 2):
            lefts, rights = self.allPossibleFBT(i), self.allPossibleFBT(n - 1 - i)
            for l in lefts:
                for r in rights:
                    res.append(TreeNode(0, l, r))
        return res

sol_rec = Solution().allPossibleFBT
for n, expected in examples:
    assert sorted(serialize(t) for t in sol_rec(n)) == sorted(serialize(build_tree(lst)) for lst in expected)
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `F(k)` bị tính lại nhiều lần nên thời gian lớn hơn Cách 3 theo một hệ số tăng theo `n` (vẫn chạy được với `n <= 19`); bộ nhớ gồm cả các bản tính lại.

---

### 7.2. Cách 3 — Ghi nhớ `F(k)` (cách nộp chính)

**Ý tưởng khác.** `F(k)` chỉ phụ thuộc `k`, nên ta **ghi nhớ** (`@cache`) kết quả. Mỗi `F(k)` chỉ được tính **một lần**; các cây lớn hơn **dùng chung** (chia sẻ) các cây con đã tính. Kết quả cuối là danh sách các cây có nút chung: về mặt cấu trúc vẫn là `C_m` cây hợp lệ (bộ chấm chỉ đọc cấu trúc).

**Giải thích từng đoạn.**

- `@cache` trên hàm `dfs(n)`.
- Hai cây khác nhau trong kết quả có thể chứa **cùng một đối tượng** `TreeNode` (cây con chung): tiết kiệm bộ nhớ, nhưng **không được sửa** nút sau khi trả về (xem lỗi 2).

In [10]:
def fbt_memo(n: int) -> List[TreeNode]:
    @cache
    def dfs(k: int) -> tuple:
        if k % 2 == 0:
            return ()
        if k == 1:
            return (TreeNode(),)
        res = []
        for i in range(1, k - 1, 2):
            for l in dfs(i):
                for r in dfs(k - 1 - i):
                    res.append(TreeNode(0, l, r))
        return tuple(res)

    return list(dfs(n))

for n, expected in examples:
    got = fbt_memo(n)
    assert sorted(serialize(t) for t in got) == sorted(serialize(build_tree(lst)) for lst in expected)
    print(f"n={n}: {len(got)} cây khớp ví dụ")

n=7: 5 cây khớp ví dụ
n=3: 1 cây khớp ví dụ


**Code nộp LeetCode (Cách 3):**

In [11]:
class Solution:
    def allPossibleFBT(self, n: int) -> List[Optional[TreeNode]]:
        @cache
        def dfs(k: int) -> tuple:
            if k % 2 == 0:
                return ()
            if k == 1:
                return (TreeNode(),)
            res = []
            for i in range(1, k - 1, 2):
                for l in dfs(i):
                    for r in dfs(k - 1 - i):
                        res.append(TreeNode(0, l, r))
            return tuple(res)

        return list(dfs(n))

sol_memo = Solution().allPossibleFBT
for n, expected in examples:
    assert sorted(serialize(t) for t in sol_memo(n)) == sorted(serialize(build_tree(lst)) for lst in expected)
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(tổng số cây cần tạo)`: mỗi cây trong `F(k)` là một `TreeNode` mới chỉ với `O(1)` công việc (hai con trỏ đến cây con có sẵn); `F(n)` có `C_m` phần tử, các `F(k)` nhỏ hơn có tổng cỡ `O(C_m)` nên tổng `O(C_m * m)` hoặc tốt hơn. Bộ nhớ `O(tổng số nút mới)`, nhỏ hơn nhiều so với không chia sẻ (`O(C_m * n)`).

---

### 7.3. Cách 4 — Quy hoạch động từ dưới lên

**Ý tưởng khác.** Tính lần lượt `dp[1], dp[3], dp[5], ..., dp[n]` bằng cùng công thức nhưng **không dùng đệ quy**: `dp[k]` được ghép từ `dp[i]` và `dp[k - 1 - i]` với `i` lẻ nhỏ hơn `k - 1`. Tránh giới hạn đệ quy (không có vấn đề với `n <= 20`) và rõ ràng về thứ tự tính.

**Giải thích từng đoạn.** `dp = {1: [TreeNode()]}`; với `k = 3, 5, ..., n`: `dp[k] = [TreeNode(0, l, r) for i in range(1, k - 1, 2) for l in dp[i] for r in dp[k - 1 - i]]`.

In [12]:
def fbt_dp(n: int) -> List[TreeNode]:
    if n % 2 == 0:
        return []
    dp = {1: [TreeNode()]}
    for k in range(3, n + 1, 2):
        dp[k] = [TreeNode(0, l, r) for i in range(1, k - 1, 2) for l in dp[i] for r in dp[k - 1 - i]]
    return dp[n]

for n, expected in examples:
    got = fbt_dp(n)
    assert sorted(serialize(t) for t in got) == sorted(serialize(build_tree(lst)) for lst in expected)
    print(f"n={n}: {len(got)} cây khớp ví dụ")

n=7: 5 cây khớp ví dụ
n=3: 1 cây khớp ví dụ


**Code nộp LeetCode (Cách 4):**

In [13]:
class Solution:
    def allPossibleFBT(self, n: int) -> List[Optional[TreeNode]]:
        if n % 2 == 0:
            return []
        dp = {1: [TreeNode()]}
        for k in range(3, n + 1, 2):
            dp[k] = [TreeNode(0, l, r) for i in range(1, k - 1, 2) for l in dp[i] for r in dp[k - 1 - i]]
        return dp[n]

sol_dp = Solution().allPossibleFBT
for n, expected in examples:
    assert sorted(serialize(t) for t in sol_dp(n)) == sorted(serialize(build_tree(lst)) for lst in expected)
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** giống Cách 3: `O(tổng số cây)` thời gian và bộ nhớ.

## 8. TRACE từng bước

### 8.1. Cách 3 cho `n = 5`: các lời gọi và kết quả

`dfs(5)`: `i = 1` (`dfs(1)` x `dfs(3)`) và `i = 3` (`dfs(3)` x `dfs(1)`). `dfs(3)` có `i = 1`: `dfs(1)` x `dfs(1)`.

In [14]:
def trace_memo(n):
    memo = {}
    def dfs(k, depth=0):
        pad = "  " * depth
        if k in memo:
            print(f"{pad}dfs({k}): đã nhớ -> {len(memo[k])} cây")
            return memo[k]
        if k % 2 == 0:
            print(f"{pad}dfs({k}): chẵn -> []")
            return []
        if k == 1:
            memo[k] = [TreeNode()]
            print(f"{pad}dfs(1): một nút -> 1 cây")
            return memo[k]
        print(f"{pad}dfs({k}): thử các kích thước cây trái {list(range(1, k - 1, 2))}")
        res = []
        for i in range(1, k - 1, 2):
            L, R = dfs(i, depth + 1), dfs(k - 1 - i, depth + 1)
            print(f"{pad}  i={i}: {len(L)} x {len(R)} = {len(L) * len(R)} cây")
            for l in L:
                for r in R:
                    res.append(TreeNode(0, l, r))
        memo[k] = res
        print(f"{pad}dfs({k}) -> {len(res)} cây")
        return res
    return dfs(n)

res = trace_memo(5)
print([tree_to_list(t) for t in res])
assert sorted(map(serialize, res)) == sorted(map(serialize, fbt_dp(5)))

dfs(5): thử các kích thước cây trái [1, 3]
  dfs(1): một nút -> 1 cây
  dfs(3): thử các kích thước cây trái [1]
    dfs(1): đã nhớ -> 1 cây
    dfs(1): đã nhớ -> 1 cây
    i=1: 1 x 1 = 1 cây
  dfs(3) -> 1 cây
  i=1: 1 x 1 = 1 cây
  dfs(3): đã nhớ -> 1 cây
  dfs(1): đã nhớ -> 1 cây
  i=3: 1 x 1 = 1 cây
dfs(5) -> 2 cây
[[0, 0, 0, None, None, 0, 0], [0, 0, 0, 0, 0]]


### 8.2. Cách 4 (từ dưới lên) cho `n = 7`: số cây của từng `dp[k]`

In [15]:
def trace_dp(n):
    dp = {1: [TreeNode()]}
    print("dp[1] = 1 cây")
    for k in range(3, n + 1, 2):
        parts = []
        dp[k] = []
        for i in range(1, k - 1, 2):
            c = len(dp[i]) * len(dp[k - 1 - i])
            parts.append(f"{len(dp[i])}x{len(dp[k - 1 - i])}")
            dp[k] += [TreeNode(0, l, r) for l in dp[i] for r in dp[k - 1 - i]]
        print(f"dp[{k}] = " + " + ".join(parts) + f" = {len(dp[k])} cây")
    return dp[n]

assert len(trace_dp(7)) == 5

dp[1] = 1 cây
dp[3] = 1x1 = 1 cây
dp[5] = 1x1 + 1x1 = 2 cây
dp[7] = 1x2 + 1x1 + 2x1 = 5 cây


### 8.3. Danh sách theo mức của 5 cây với `n = 7`

In [16]:
for t in sorted(sol_memo(7), key=serialize):
    print(tree_to_list(t))

[0, 0, 0, None, None, 0, 0, None, None, 0, 0]
[0, 0, 0, None, None, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, None, None, None, None, 0, 0]
[0, 0, 0, 0, 0, None, None, 0, 0]


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (đệ quy có ghi nhớ): gọn, đúng công thức chia để trị, mỗi `F(k)` chỉ tính một lần và các cây dùng chung cây con.

```python
class Solution:
    def allPossibleFBT(self, n: int) -> List[Optional[TreeNode]]:
        @cache
        def dfs(k: int) -> tuple:
            if k % 2 == 0:
                return ()
            if k == 1:
                return (TreeNode(),)
            res = []
            for i in range(1, k - 1, 2):
                for l in dfs(i):
                    for r in dfs(k - 1 - i):
                        res.append(TreeNode(0, l, r))
            return tuple(res)

        return list(dfs(n))
```

In [17]:
class Solution:
    def allPossibleFBT(self, n: int) -> List[Optional[TreeNode]]:
        @cache
        def dfs(k: int) -> tuple:
            if k % 2 == 0:
                return ()
            if k == 1:
                return (TreeNode(),)
            res = []
            for i in range(1, k - 1, 2):
                for l in dfs(i):
                    for r in dfs(k - 1 - i):
                        res.append(TreeNode(0, l, r))
            return tuple(res)

        return list(dfs(n))

sol = Solution()
for n, expected in examples:
    got = sol.allPossibleFBT(n)
    ok = sorted(serialize(t) for t in got) == sorted(serialize(build_tree(lst)) for lst in expected)
    print(n, "->", len(got), "cây", "OK" if ok else "SAI")
    assert ok

7 -> 5 cây OK
3 -> 1 cây OK


## 10. Vì sao đúng

**(a) Cấu trúc.** Cây đầy đủ `n > 1` nút có gốc với **đúng hai** con, mỗi con là gốc của một cây đầy đủ. Gọi `i` là số nút của cây trái; số nút cây phải là `n - 1 - i`. Mỗi cây con đầy đủ có số nút lẻ, nên `i` lẻ, `n - 1 - i` lẻ, suy ra `n = 1 + i + (n - 1 - i)` lẻ; với `n` chẵn không có cây nào.

**(b) Song ánh.** Ánh xạ `cây -> (i, cây trái, cây phải)` là **song ánh** giữa `F(n)` và tập các bộ `(i, l, r)` với `i` lẻ trong `[1, n - 2]`, `l in F(i)`, `r in F(n - 1 - i)`: cây được xác định duy nhất bởi bộ ba và ngược lại mọi bộ ba cho một cây đầy đủ `n` nút. Vì vậy thuật toán liệt kê **mỗi cây đúng một lần**, không trùng, không sót.

**(c) Quy nạp.** `F(1)` đúng (một nút). Giả sử `F(k)` đúng với mọi `k < n`; theo (b) `F(n)` được sinh đúng.

**(d) Số cây.** Từ (b): `|F(n)| = sum_i |F(i)| * |F(n - 1 - i)|` với `|F(1)| = 1`: đây chính là hệ thức đệ quy của số Catalan: `|F(2m + 1)| = C_m`.

**(e) Chia sẻ cây con (Cách 3, 4).** Hai cây khác nhau trong `F(n)` có thể chung các nút của cây con. Điều đó **không** ảnh hưởng tính đúng cấu trúc (mỗi cây vẫn là cây có gốc riêng với cấu trúc đúng), nhưng nếu sửa nút ở cây này thì ảnh hưởng cây kia (lỗi 2).

**Kiểm chứng bằng máy.** Dựng hai oracle **độc lập với công thức chia để trị**: (1) **liệt kê mọi hình dạng cây nhị phân** rồi lọc cây đầy đủ (`n <= 13`); (2) **mở rộng lá**: bắt đầu từ cây một nút; mỗi bước chọn **một lá** rồi biến nó thành nút trong có hai lá con; lặp `(n - 1) / 2` lần, gom các hình dạng vào `set` (mỗi cây đầy đủ nhận được theo cách này), dùng cho mọi `n` tới `19`. Kết quả mỗi lời giải phải: (i) có đúng `C_m` cây; (ii) mỗi cây đầy đủ, đủ `n` nút, mọi giá trị `0`; (iii) các cây **đôi một khác nhau**; (iv) tập hình dạng bằng tập của oracle.

In [18]:
def leaf_expansion_oracle(n: int) -> set:
    """Tập hình dạng cây đầy đủ n nút, bằng cách mở rộng từng lá (n lẻ)."""
    if n % 2 == 0:
        return set()
    def expansions(sh):
        l, r = sh if sh is not None else (None, None)
        if l is None and r is None:                    # lá -> nút trong có hai lá con
            yield ((None, None), (None, None))
            return
        for nl in expansions(l):
            yield (nl, r)
        for nr in expansions(r):
            yield (l, nr)
    level = {(None, None)}                              # cây một nút
    for _ in range((n - 1) // 2):
        level = {nxt for sh in level for nxt in expansions(sh)}
    return level

def check_result(n: int, trees, oracle: set) -> None:
    m = (n - 1) // 2 if n % 2 == 1 else None
    assert isinstance(trees, list)
    if n % 2 == 0:
        assert trees == [], n
        return
    assert len(trees) == catalan(m), (n, len(trees), catalan(m))
    shapes_seen = set()
    for t in trees:
        assert t is not None and is_full(t) and count_nodes(t) == n
        stack = [t]
        while stack:                                    # mọi giá trị phải bằng 0
            x = stack.pop()
            assert x.val == 0
            if x.left: stack.append(x.left)
            if x.right: stack.append(x.right)
        shapes_seen.add(to_shape(t))
    assert len(shapes_seen) == len(trees), "có cây trùng nhau"
    assert shapes_seen == oracle, "tập hình dạng khác oracle"

# hai oracle độc lập khớp nhau và khớp công thức Catalan
for n in range(1, 14):
    brute = full_shapes_brute(n) if n % 2 == 1 else set()
    exp = leaf_expansion_oracle(n)
    assert brute == exp, n
    if n % 2 == 1:
        assert len(exp) == catalan((n - 1) // 2)
for n in range(15, 20, 2):
    assert len(leaf_expansion_oracle(n)) == catalan((n - 1) // 2)
print("oracle liệt kê hình dạng và oracle mở rộng lá khớp nhau (n <= 13) và khớp số Catalan (n tới 19)")

oracle liệt kê hình dạng và oracle mở rộng lá khớp nhau (n <= 13) và khớp số Catalan (n tới 19)


## 11. Time / Space Complexity

Gọi `m = (n - 1) / 2` (số nút trong), `C_m` là số Catalan.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi hình dạng + lọc | `O(C_n * n)` (rất lớn) | `O(C_n * n)` |
| 2. Đệ quy không nhớ | `O(số lần dựng lại * C_m)` | `O(C_m * n)` nếu không chia sẻ |
| 3. Đệ quy có nhớ | `O(tổng số cây)` `~ O(C_m)` | `O(C_m)` nút mới nhờ chia sẻ |
| 4. DP từ dưới lên | `O(tổng số cây)` | `O(C_m)` |

Số cây và thời gian thực tế:

In [19]:
for n in (1, 3, 5, 7, 9, 11, 13, 15, 17, 19):
    row = [f"n={n:2} ({catalan((n - 1) // 2):5} cây)"]
    for name, fn in (("Cách 2", sol_rec), ("Cách 3", Solution().allPossibleFBT), ("Cách 4", sol_dp)):
        t0 = time.perf_counter()
        res = fn(n)
        row.append(f"{name}: {1000 * (time.perf_counter() - t0):8.2f} ms")
        assert len(res) == catalan((n - 1) // 2)
    print(" | ".join(row))

n= 1 (    1 cây) | Cách 2:     0.00 ms | Cách 3:     0.02 ms | Cách 4:     0.00 ms
n= 3 (    1 cây) | Cách 2:     0.00 ms | Cách 3:     0.01 ms | Cách 4:     0.00 ms
n= 5 (    2 cây) | Cách 2:     0.01 ms | Cách 3:     0.01 ms | Cách 4:     0.00 ms
n= 7 (    5 cây) | Cách 2:     0.01 ms | Cách 3:     0.01 ms | Cách 4:     0.00 ms
n= 9 (   14 cây) | Cách 2:     0.03 ms | Cách 3:     0.01 ms | Cách 4:     0.01 ms
n=11 (   42 cây) | Cách 2:     0.07 ms | Cách 3:     0.02 ms | Cách 4:     0.01 ms
n=13 (  132 cây) | Cách 2:     0.22 ms | Cách 3:     0.09 ms | Cách 4:     0.03 ms
n=15 (  429 cây) | Cách 2:     1.06 ms | Cách 3:     0.17 ms | Cách 4:     0.07 ms
n=17 ( 1430 cây) | Cách 2:     3.87 ms | Cách 3:     0.69 ms | Cách 4:     0.37 ms


n=19 ( 4862 cây) | Cách 2:    13.80 ms | Cách 3:     5.32 ms | Cách 4:     1.35 ms


## 12. Bộ test

Gồm: ví dụ của đề (so sánh danh sách theo mức đã sắp xếp), edge case, phản ví dụ cho các cách sai, đối chiếu mọi cách với hai oracle (vét cạn hình dạng và mở rộng lá), kiểm tra cấu trúc (đầy đủ, đủ `n` nút, giá trị `0`, đôi một khác nhau), số lượng theo Catalan và tính chất. Thứ tự kết quả không quan trọng nên mọi so sánh đều qua `sorted` / `set`.

In [20]:
def solutions_list():
    return {
        "recursive": fbt_recursive,
        "memo": fbt_memo,
        "dp": fbt_dp,
        "sol_rec": sol_rec,
        "sol_memo": sol_memo,
        "sol_dp": sol_dp,
        "final": Solution().allPossibleFBT,
    }

def run_all(n: int):
    oracle = leaf_expansion_oracle(n)
    for name, fn in solutions_list().items():
        trees = fn(n)
        check_result(n, trees, oracle)
    return len(oracle)

# 12.1 ví dụ đề: so sánh danh sách theo mức đã sắp xếp (một chuỗi so sánh được cho mỗi cây)
def level_key(lst):
    return tuple(-1 if v is None else v for v in lst)

for n, expected in examples:
    for name, fn in solutions_list().items():
        got = sorted(level_key(tree_to_list(t)) for t in fn(n))
        assert got == sorted(level_key(lst) for lst in expected), (name, n)
    assert run_all(n) == len(expected)
print("12.1 ví dụ đề (so với danh sách theo mức của đề): OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    (1, 1),        # một nút
    (2, 0),        # chẵn
    (3, 1),        # gốc + hai lá
    (4, 0),
    (5, 2),
    (6, 0),
    (9, 14),       # Catalan thứ 4
    (11, 42),
    (15, 429),
    (19, 4862),    # Catalan thứ 9
    (20, 0),       # giới hạn trên của đề, chẵn
]
for n, expected in edge:
    assert run_all(n) == expected, n
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề (so với danh sách theo mức của đề): OK


12.2 edge case: OK 11 ca


In [21]:
# 12.3 phản ví dụ: các cách sai (xác nhận lời giải đúng không bị lỗi trên chính ca đó)
def wrong_allow_empty(n):                          # cho phép cây con rỗng (i từ 0): sinh nút một con
    @cache
    def dfs(k):
        if k == 0:
            return (None,)
        if k == 1:
            return (TreeNode(),)
        res = []
        for i in range(0, k):
            for l in dfs(i):
                for r in dfs(k - 1 - i):
                    res.append(TreeNode(0, l, r))
        return tuple(res)
    return list(dfs(n))

def wrong_range(n):                                # cận vòng lặp sai: range(1, k - 2, 2) bỏ sót kích thước
    @cache
    def dfs(k):
        if k % 2 == 0:
            return ()
        if k == 1:
            return (TreeNode(),)
        res = []
        for i in range(1, k - 2, 2):
            for l in dfs(i):
                for r in dfs(k - 1 - i):
                    res.append(TreeNode(0, l, r))
        return tuple(res)
    return list(dfs(n))

def wrong_even_not_empty(n):                       # không xử lý n chẵn: coi n chẵn như n - 1
    return Solution().allPossibleFBT(n - 1 if n % 2 == 0 else n)

def wrong_only_left_heavy(n):                      # chỉ xét cây trái lớn hơn hoặc bằng cây phải (bỏ đối xứng)
    @cache
    def dfs(k):
        if k % 2 == 0:
            return ()
        if k == 1:
            return (TreeNode(),)
        res = []
        for i in range(1, k - 1, 2):
            if i < k - 1 - i:
                continue
            for l in dfs(i):
                for r in dfs(k - 1 - i):
                    res.append(TreeNode(0, l, r))
        return tuple(res)
    return list(dfs(n))

def verdict(fn, n):
    try:
        trees = fn(n)
        check_result(n, trees, leaf_expansion_oracle(n))
        return "đúng"
    except AssertionError:
        return "SAI"

counters = {}
for name, fn in (("cho phép cây con rỗng", wrong_allow_empty), ("cận vòng lặp sai", wrong_range),
                 ("không xử lý n chẵn", wrong_even_not_empty), ("chỉ xét cây trái lớn hơn", wrong_only_left_heavy)):
    bad = next(n for n in range(1, 12) if verdict(fn, n) == "SAI")
    counters[name] = bad
    print(f"{name:28} n={bad}: lời giải sai trả {len(fn(bad))} cây, đúng là {len(leaf_expansion_oracle(bad))}")
    assert run_all(bad) == len(leaf_expansion_oracle(bad))
print("12.3 phản ví dụ: OK")

cho phép cây con rỗng        n=2: lời giải sai trả 2 cây, đúng là 0
cận vòng lặp sai             n=3: lời giải sai trả 0 cây, đúng là 1
không xử lý n chẵn           n=2: lời giải sai trả 1 cây, đúng là 0
chỉ xét cây trái lớn hơn     n=5: lời giải sai trả 1 cây, đúng là 2
12.3 phản ví dụ: OK


In [22]:
# 12.4 mọi n từ 1 đến 20: mọi lời giải khớp oracle mở rộng lá (so với oracle liệt kê hình dạng cho n <= 13)
for n in range(1, 21):
    run_all(n)
    if n <= 13:
        assert full_shapes_brute(n) == leaf_expansion_oracle(n)
print("12.4 mọi n = 1..20 qua hai oracle và kiểm tra cấu trúc: OK")

# 12.5 mỗi cây trong kết quả khi dựng lại từ danh sách theo mức cho đúng hình dạng cũ (tuần tự hóa hai chiều)
for n in (1, 3, 5, 7, 9, 11):
    for t in Solution().allPossibleFBT(n):
        t2 = build_tree(tree_to_list(t))
        assert serialize(t) == serialize(t2) and tree_to_list(t) == tree_to_list(t2)
print("12.5 tuần tự hóa theo mức hai chiều: OK")

# 12.6 các lời giải cho cùng tập hình dạng và cùng thứ tự đếm (so sánh tập tuần tự hóa)
for n in (5, 9, 13, 17, 19):
    sets = {name: {serialize(t) for t in fn(n)} for name, fn in solutions_list().items()}
    assert len({frozenset(s) for s in sets.values()}) == 1, n
print("12.6 mọi lời giải cho cùng một tập cây: OK")

# 12.7 tính chất: tổng số nút lá của mọi cây n nút là (n + 1) / 2 và số nút trong (n - 1) / 2
def count_leaves(t):
    return 1 if t.left is None and t.right is None else count_leaves(t.left) + count_leaves(t.right)

for n in (1, 3, 5, 7, 9, 11, 13):
    for t in Solution().allPossibleFBT(n):
        assert count_leaves(t) == (n + 1) // 2
# đối xứng gương: lật trái-phải một cây đầy đủ vẫn là cây đầy đủ trong tập kết quả
def mirror(t):
    return None if t is None else TreeNode(t.val, mirror(t.right), mirror(t.left))

for n in (5, 7, 9, 11):
    trees = Solution().allPossibleFBT(n)
    keys = {serialize(t) for t in trees}
    assert all(serialize(mirror(t)) in keys for t in trees)
print("12.7 tính chất (số lá, đối xứng gương): OK")

12.4 mọi n = 1..20 qua hai oracle và kiểm tra cấu trúc: OK
12.5 tuần tự hóa theo mức hai chiều: OK
12.6 mọi lời giải cho cùng một tập cây: OK
12.7 tính chất (số lá, đối xứng gương): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép cây con rỗng (tạo nút một con)

Nếu vòng lặp `i` bắt đầu từ `0` (cây trái rỗng) hoặc kết thúc ở `n - 1` (cây phải rỗng) thì xuất hiện **nút chỉ có một con**, vi phạm "mỗi nút có 0 hoặc 2 con".

In [23]:
n = counters["cho phép cây con rỗng"]
bad = wrong_allow_empty(n)
print(f"n = {n}: lời giải sai trả {len(bad)} cây; đúng là {len(Solution().allPossibleFBT(n))}")
print("có cây không đầy đủ?", any(t is not None and not is_full(t) for t in bad))
assert any(t is not None and not is_full(t) for t in bad)

n = 2: lời giải sai trả 2 cây; đúng là 0
có cây không đầy đủ? True


Vì sao sai: `i = 0` ghép cây trái rỗng với cây phải `n - 1` nút, cho gốc có một con. **Đúng:** `i` chạy qua các số **lẻ** từ `1` đến `n - 2` (`range(1, n - 1, 2)`).

### Lỗi 2 — Sửa nút khi các cây chia sẻ cây con

Với Cách 3/4, nhiều cây dùng **chung** đối tượng cây con; sửa một nút ở cây này làm đổi cả các cây khác (ngoài hậu quả với bộ chấm, đây là lỗi tinh vi khi bạn dùng kết quả cho việc khác).

In [24]:
trees = Solution().allPossibleFBT(7)
before = sorted(serialize(t) for t in trees)
leaf = trees[0]
while leaf.left is not None:
    leaf = leaf.left
leaf.val = 7                                           # chỉ định sửa MỘT lá của cây đầu tiên
after = sorted(serialize(t) for t in trees)
affected = sum(1 for t in trees if "7" in serialize(t))
print(f"sau khi sửa một lá: {affected} cây trong danh sách chứa giá trị 7 (mong đợi 1)")
assert affected > 1

sau khi sửa một lá: 5 cây trong danh sách chứa giá trị 7 (mong đợi 1)


Vì sao: lá đó là **cùng một đối tượng** trong nhiều cây. **Đúng:** nếu cần sửa, hãy **sao chép sâu** cây (`copy.deepcopy`) hoặc dựng cây mới cho từng kết quả (Cách 2 vẫn chia sẻ trong cùng một lần gọi; muốn tránh hoàn toàn phải sao chép từng cây).

### Lỗi 3 — Cận vòng lặp sai

`range(1, k - 2, 2)` bỏ sót kích thước cây trái lớn nhất `k - 2`.

In [25]:
n = counters["cận vòng lặp sai"]
print(f"n = {n}: lời giải sai trả {len(wrong_range(n))} cây; đúng là {len(Solution().allPossibleFBT(n))}")
assert len(wrong_range(n)) != len(Solution().allPossibleFBT(n))

n = 3: lời giải sai trả 0 cây; đúng là 1


Vì sao sai: bỏ sót các cây có cây phải chỉ một nút. **Đúng:** `range(1, n - 1, 2)` (hoặc `range(1, n, 2)` vì `n` lẻ).

### Lỗi 4 — Quên xử lý `n` chẵn

Với `n` chẵn không có cây nào, phải trả `[]`; coi `n` chẵn như `n - 1` hoặc để rơi vào đệ quy vô nghĩa cho kết quả sai.

In [26]:
n = counters["không xử lý n chẵn"]
print(f"n = {n}: lời giải sai trả {len(wrong_even_not_empty(n))} cây; đúng là {len(Solution().allPossibleFBT(n))} (danh sách rỗng)")
assert Solution().allPossibleFBT(n) == [] and len(wrong_even_not_empty(n)) > 0

n = 2: lời giải sai trả 1 cây; đúng là 0 (danh sách rỗng)


Vì sao sai: số nút của cây đầy đủ luôn lẻ. **Đúng:** `if n % 2 == 0: return []` ở đầu.

### Lỗi 5 — So sánh cây bằng `==` (hoặc kỳ vọng thứ tự cố định)

`TreeNode` không có `__eq__` nên hai cây cùng cấu trúc vẫn "khác nhau"; kết quả cũng có thể ở thứ tự bất kỳ.

In [27]:
x, y = tree_with_internal(2), tree_with_internal(2)
print("x == y :", x == y, "| serialize bằng nhau:", serialize(x) == serialize(y))
a, b = Solution().allPossibleFBT(5), fbt_dp(5)
print("so sánh danh sách đối tượng bằng ==:", a == b, "| so sánh tập tuần tự hóa:", {serialize(t) for t in a} == {serialize(t) for t in b})
assert x != y and serialize(x) == serialize(y)
assert a != b and {serialize(t) for t in a} == {serialize(t) for t in b}

x == y : False | serialize bằng nhau: True
so sánh danh sách đối tượng bằng ==: False | so sánh tập tuần tự hóa: True


Vì sao: `==` mặc định so sánh **danh tính** đối tượng. **Đúng:** tuần tự hóa (hoặc xuất danh sách theo mức) rồi so sánh sau khi `sorted`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "liệt kê mọi cấu trúc cây" ⇒ **chia để trị theo gốc**: `F(n) = ghép mọi cặp (F(i), F(n-1-i))`; thường kết hợp **ghi nhớ**.
- **Ràng buộc chẵn/lẻ:** cây đầy đủ có số nút lẻ ⇒ lọc sớm, vòng lặp bước `2`.
- **Số lượng** là số Catalan: dùng để kiểm tra nhanh kết quả.
- **Chia sẻ cây con** tiết kiệm bộ nhớ nhưng cấm sửa nút.
- Cùng họ: Unique Binary Search Trees II (95), Unique Binary Search Trees (96), Generate Parentheses (22), Different Ways to Add Parentheses (241).
- Test nên có: `n = 1`, `n` chẵn, `n = 19` (4862 cây), kiểm tra bằng tuần tự hóa và số Catalan.

## 15. Biến thể

1. **Chỉ đếm số cây** (bài tập 5): `count(n) = sum count(i) * count(n - 1 - i)`, kết quả là số Catalan, tính được cho `n` rất lớn.
2. **Liệt kê cây có độ cao nhỏ hơn `h`** hoặc **có đúng `k` lá**: đổi tham số trạng thái.
3. **Unique Binary Search Trees II (95):** cây nhị phân tìm kiếm với giá trị `1..n`, cùng khung ghép trái x phải.
4. Bài liên quan: Generate Parentheses (22).

Dưới đây là biến thể 1 (đếm), kiểm bằng oracle và công thức Catalan.

In [28]:
def count_full_trees(n: int) -> int:
    if n % 2 == 0:
        return 0
    dp = {0: 0, 1: 1}
    for k in range(3, n + 1, 2):
        dp[k] = sum(dp[i] * dp[k - 1 - i] for i in range(1, k - 1, 2))
    return dp[n]

for n in range(1, 40):
    assert count_full_trees(n) == (catalan((n - 1) // 2) if n % 2 else 0)
for n in range(1, 20):
    assert count_full_trees(n) == len(leaf_expansion_oracle(n))
print([count_full_trees(n) for n in (1, 3, 5, 7, 9, 11)], "| n = 99:", count_full_trees(99))

[1, 1, 2, 5, 14, 42] | n = 99: 509552245179617138054608572


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi hình dạng + lọc | Định nghĩa | `O(C_n n)` | `O(C_n n)` | Oracle, `n <= 13` |
| 2. Đệ quy không nhớ | Công thức `F(n)` | dựng lại nhiều lần | `O(C_m n)` | Dễ hiểu |
| 3. Đệ quy có nhớ | `@cache` + chia sẻ | `~ O(C_m)` | `~ O(C_m)` | **Cách nộp chính** |
| 4. DP từ dưới lên | Tính `F(1), F(3), ...` | `~ O(C_m)` | `~ O(C_m)` | Không đệ quy |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
print(list(range(1, 9, 2)))
print([len(Solution().allPossibleFBT(n)) for n in (1, 2, 3, 5, 7)])
print(tree_to_list(tree_with_internal(2)))
```

In [29]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `n = 5`: ghi hai cây đầy đủ dưới dạng danh sách theo mức (kiểu LeetCode, dùng `None`).

In [30]:
# Bài 2: điền kết quả tính tay
hand_trees = None             # ví dụ [[0, 0, 0, ...], [0, 0, 0, ...]]

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành đệ quy có ghi nhớ.

In [31]:
def exercise_3_memo(n: int) -> List[TreeNode]:
    @cache
    def dfs(k: int) -> tuple:
        if k % 2 == 0:
            return ()
        if k == 1:
            return (TreeNode(),)
        res = []
        # TODO: với mỗi i lẻ từ 1 đến k - 2, ghép mọi cặp (l, r) với l thuộc dfs(i), r thuộc dfs(k - 1 - i)
        #       và thêm TreeNode(0, l, r) vào res
        return tuple(res)

    return list(dfs(n))


# assert len(exercise_3_memo(7)) == 5
# assert len(exercise_3_memo(3)) == 1
# assert exercise_3_memo(4) == []

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_dp(n)` bằng quy hoạch động từ dưới lên.

In [32]:
def exercise_4_dp(n: int) -> List[TreeNode]:
    # TODO
    pass


# assert len(exercise_4_dp(7)) == 5
# assert len(exercise_4_dp(9)) == 14
# assert exercise_4_dp(6) == []

### Bài 5 — Mini challenge: chỉ đếm số cây (khó)

Viết `exercise_5_count(n)` trả về **số** cây nhị phân đầy đủ có `n` nút **không** dựng cây, và chạy được cả khi `n` lớn (ví dụ `n = 99`).

```python
exercise_5_count(1)   -> 1
exercise_5_count(2)   -> 0
exercise_5_count(7)   -> 5
exercise_5_count(19)  -> 4862
```
Gợi ý: cùng công thức chia để trị nhưng cộng tích các số đếm.

In [33]:
def exercise_5_count(n: int) -> int:
    # TODO
    pass


# assert exercise_5_count(1) == 1
# assert exercise_5_count(2) == 0
# assert exercise_5_count(7) == 5
# assert exercise_5_count(19) == 4862

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `range(1, 9, 2)` là các số lẻ nhỏ hơn 9.
- Mức 2: số cây đầy đủ với `n = 1, 3, 5, 7` là `1, 1, 2, 5`; `n = 2` là `0`.
- Mức 3: `[1, 3, 5, 7]`, `[1, 0, 1, 2, 5]`, `[0, 0, 0, 0, 0]`.

**Bài 2**
- Mức 1: chia `4` nút còn lại thành (trái, phải) lẻ: `(1, 3)` hoặc `(3, 1)`.
- Mức 2: cây phải là cây 3 nút (gốc + hai lá) hoặc cây trái là cây 3 nút.
- Mức 3: `[0, 0, 0, None, None, 0, 0]` và `[0, 0, 0, 0, 0]`.

**Bài 3**
- Mức 1: hai vòng `for` lồng nhau qua `dfs(i)` và `dfs(k - 1 - i)`.
- Mức 2: `for i in range(1, k - 1, 2):`.
- Mức 3: `res.append(TreeNode(0, l, r))` trong vòng lặp trong cùng.

**Bài 4**
- Mức 1: `dp = {1: [TreeNode()]}`.
- Mức 2: `for k in range(3, n + 1, 2)`.
- Mức 3: `dp[k] = [TreeNode(0, l, r) for i in range(1, k - 1, 2) for l in dp[i] for r in dp[k - 1 - i]]`; nhớ `n` chẵn trả `[]`.

**Bài 5**
- Mức 1: `c[1] = 1`; `n` chẵn trả `0`.
- Mức 2: `c[k] = sum(c[i] * c[k - 1 - i])` theo `i` lẻ.
- Mức 3: kết quả là số Catalan thứ `(n - 1) / 2`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [34]:
print(list(range(1, 9, 2)))
print([len(Solution().allPossibleFBT(n)) for n in (1, 2, 3, 5, 7)])
print(tree_to_list(tree_with_internal(2)))

assert list(range(1, 9, 2)) == [1, 3, 5, 7]
assert [len(Solution().allPossibleFBT(n)) for n in (1, 2, 3, 5, 7)] == [1, 0, 1, 2, 5]
assert tree_to_list(tree_with_internal(2)) == [0, 0, 0, 0, 0]

[1, 3, 5, 7]
[1, 0, 1, 2, 5]
[0, 0, 0, 0, 0]


### Đáp án Bài 2

`(1, 3)`: gốc, trái là lá, phải là cây 3 nút `[0, 0, 0, None, None, 0, 0]`. `(3, 1)`: trái là cây 3 nút, phải là lá `[0, 0, 0, 0, 0]`.

In [35]:
hand_trees_solution = [[0, 0, 0, None, None, 0, 0], [0, 0, 0, 0, 0]]
got = sorted(level_key(tree_to_list(t)) for t in Solution().allPossibleFBT(5))
assert got == sorted(level_key(x) for x in hand_trees_solution)
for lst in hand_trees_solution:
    t = build_tree(lst)
    assert is_full(t) and count_nodes(t) == 5
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [36]:
def exercise_3_memo_solution(n: int) -> List[TreeNode]:
    @cache
    def dfs(k: int) -> tuple:
        if k % 2 == 0:
            return ()
        if k == 1:
            return (TreeNode(),)
        res = []
        for i in range(1, k - 1, 2):
            for l in dfs(i):
                for r in dfs(k - 1 - i):
                    res.append(TreeNode(0, l, r))
        return tuple(res)

    return list(dfs(n))

assert len(exercise_3_memo_solution(7)) == 5
assert len(exercise_3_memo_solution(3)) == 1
assert exercise_3_memo_solution(4) == []
for n in range(1, 20):
    check_result(n, exercise_3_memo_solution(n), leaf_expansion_oracle(n))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [37]:
def exercise_4_dp_solution(n: int) -> List[TreeNode]:
    if n % 2 == 0:
        return []
    dp = {1: [TreeNode()]}
    for k in range(3, n + 1, 2):
        dp[k] = [TreeNode(0, l, r) for i in range(1, k - 1, 2) for l in dp[i] for r in dp[k - 1 - i]]
    return dp[n]

assert len(exercise_4_dp_solution(7)) == 5
assert len(exercise_4_dp_solution(9)) == 14
assert exercise_4_dp_solution(6) == []
for n in range(1, 20):
    check_result(n, exercise_4_dp_solution(n), leaf_expansion_oracle(n))
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [38]:
def exercise_5_count_solution(n: int) -> int:
    if n % 2 == 0:
        return 0
    c = {1: 1}
    for k in range(3, n + 1, 2):
        c[k] = sum(c[i] * c[k - 1 - i] for i in range(1, k - 1, 2))
    return c[n]

assert exercise_5_count_solution(1) == 1
assert exercise_5_count_solution(2) == 0
assert exercise_5_count_solution(7) == 5
assert exercise_5_count_solution(19) == 4862
for n in range(1, 20):
    assert exercise_5_count_solution(n) == len(leaf_expansion_oracle(n))
assert exercise_5_count_solution(99) == catalan(49)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (chia để trị + ghi nhớ)

```python
class Solution:
    def allPossibleFBT(self, n: int) -> List[Optional[TreeNode]]:
        @cache
        def dfs(k: int) -> tuple:
            if k % 2 == 0:                          # số nút của cây đầy đủ luôn lẻ
                return ()
            if k == 1:                              # một nút
                return (TreeNode(),)
            res = []
            for i in range(1, k - 1, 2):            # kích thước cây trái lẻ: 1, 3, ..., k - 2
                for l in dfs(i):
                    for r in dfs(k - 1 - i):        # kích thước cây phải = k - 1 - i (cũng lẻ)
                        res.append(TreeNode(0, l, r))
            return tuple(res)

        return list(dfs(n))
```

Công thức nhớ nhanh: `F(n) = [Node(0, l, r) for i lẻ in [1, n-2] for l in F(i) for r in F(n-1-i)]`; số cây là Catalan `C_{(n-1)/2}`.

### Checklist trước khi submit

- [ ] `n` chẵn trả `[]`; `n == 1` trả một nút?
- [ ] Kích thước cây trái chạy qua các số **lẻ** từ `1` đến `n - 2` (không để cây con rỗng)?
- [ ] Mọi nút có `val = 0`?
- [ ] Biết rằng các cây có thể **chia sẻ** cây con (không sửa nút sau khi trả về)?
- [ ] Kiểm thử bằng tuần tự hóa/danh sách theo mức đã `sorted` và so số lượng với Catalan, không so `TreeNode` bằng `==`?
- [ ] Đã thử: `n = 1`, `n = 2`, `n = 3`, `n = 7` (5 cây), `n = 19` (4862 cây), `n = 20`?
- [ ] Tên method đúng `allPossibleFBT(self, n)` và trả danh sách gốc cây?